# The Hedge Fund Alpha — Corn Stress from Orbit

**A commodity desk's informational edge, built on Wherobots.**

Every Monday at 4:00 PM ET the USDA releases the Crop Progress Report.
Every second Friday of the month, WASDE hits with updated yield estimates.
By the time either lands, the market has already priced in what the
well-resourced desks have been watching for weeks: **NDVI anomalies across
the Corn Belt**. Low NDVI in July = stressed canopy = lower yield = tight
supply = long corn.

This notebook walks a commodity trading firm through the full pipeline:

1. **Define the universe** — 20 top corn-producing counties across IA, IL,
   IN, NE, MN, MO.
2. **Tap the imagery** — Sentinel-2 L2A via Wherobots' STAC reader; every
   county has ~70 cloud-screened observations per year at 10 m.
3. **Compute NDVI** — `(NIR − Red) / (NIR + Red)` per scene, zonal-stats to
   a county-week grid. (Cell notes the RasterFlow call the production
   pipeline would use; demo uses a calibrated synthetic series.)
4. **Detect stress** — compare current-year NDVI to the 5-year climatology.
   Negative anomalies in June-August reliably precede the USDA yield
   mark-down.
5. **Quantify the alpha** — naive strategy: long corn futures when the
   county-weighted anomaly turns negative, flat otherwise.
6. **Ship the signal** — GeoJSON feed the desk's dashboard consumes.

> **Demo scope.** County centroids, crop calendar, and NDVI calculation
> are real. The weekly NDVI and corn price series are synthesized to
> demonstrate the analytical workflow. The Wherobots capabilities shown
> — STAC ingest, spatial joins, Iceberg-ready output, GeoJSON export —
> are the same surface a production deployment runs on.

## 1. Setup

In [ ]:
from sedona.spark import *
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. The Universe — 20 Top Corn-Producing Counties

~2% of U.S. counties grow ~40% of the crop. Focusing the pipeline on the
top producers collapses the satellite bill without material loss of
signal. Acres planted (x1000) and centroid are per USDA NASS; we buffer
the centroid to a ~25 km AOI to approximate the cropland footprint.

In [ ]:
counties = [
    # county,           state, lon,      lat,    corn_acres_k
    ("Sioux",           "IA", -96.1800, 43.0700, 285),
    ("Plymouth",        "IA", -96.1500, 42.7100, 268),
    ("Ida",             "IA", -95.5100, 42.4000, 142),
    ("Kossuth",         "IA", -94.1300, 43.2000, 298),
    ("Hamilton",        "IA", -93.7100, 42.4400, 201),
    ("Story",           "IA", -93.4700, 42.0300, 188),
    ("McLean",          "IL", -88.8300, 40.4700, 321),
    ("Iroquois",        "IL", -87.8200, 40.7300, 258),
    ("Champaign",       "IL", -88.1900, 40.1300, 235),
    ("Livingston",      "IL", -88.5500, 40.8900, 241),
    ("Benton",          "IN", -87.3000, 40.6100, 166),
    ("White",           "IN", -86.8700, 40.7500, 153),
    ("Jasper",          "IN", -87.1200, 40.9900, 141),
    ("Holt",            "NE", -98.7900, 42.4400, 224),
    ("Cuming",          "NE", -96.8800, 41.9200, 182),
    ("Clay",            "NE", -97.9600, 40.5200, 164),
    ("Butler",          "NE", -97.1300, 41.2300, 176),
    ("Nicollet",        "MN", -94.2500, 44.3500, 132),
    ("Martin",          "MN", -94.5500, 43.6700, 148),
    ("Nodaway",         "MO", -94.8800, 40.3700, 138),
]

county_schema = StructType([
    StructField("county",       StringType()),
    StructField("state",        StringType()),
    StructField("lon",          DoubleType()),
    StructField("lat",          DoubleType()),
    StructField("corn_acres_k", DoubleType()),
])

counties_df = sedona.createDataFrame(
    [(c, s, ln, lt, float(a)) for c, s, ln, lt, a in counties],
    county_schema
).withColumn(
    "aoi",
    f.expr("ST_Buffer(ST_Point(lon, lat), 0.25)")   # ~25 km radius
)
counties_df.createOrReplaceTempView("counties")

totals = sedona.sql("""
    SELECT COUNT(*) AS n, ROUND(SUM(corn_acres_k)/1000.0, 1) AS corn_acres_M
    FROM counties
""").toPandas().iloc[0]

print(f"Universe: {int(totals.n)} counties, "
      f"{totals.corn_acres_M:.1f}M acres planted")
counties_df.select("county", "state", "corn_acres_k").show(truncate=False)

## 3. Tap the Imagery — Sentinel-2 over the Corn Belt

Sentinel-2 L2A gives free 10 m red/NIR bands on a 5-day revisit. A single
growing-season query over one county returns ~35 cloud-screened scenes —
enough for a dense weekly NDVI series. Below we hit the public Element 84
STAC catalog for one representative county (McLean, IL) to show the
data supply is real.

In [ ]:
from sedona.stac.client import Client

mclean_bbox = [-89.05, 40.25, -88.55, 40.70]
stac = Client.open("https://earth-search.aws.element84.com/v1")

# The Sedona STAC client wraps endpoint errors in a generic RuntimeError;
# retry across datetime formats before giving up.
items_list = []
for dt_expr in [
    "2025-04-01T00:00:00Z/2025-10-31T00:00:00Z",
    "2025-04-01/2025-10-31",
    "2025",
]:
    try:
        items = stac.search(
            collection_id="sentinel-2-c1-l2a",
            bbox=mclean_bbox,
            datetime=dt_expr,
            max_items=200,
            return_dataframe=False,
        )
        items_list = list(items)
        print(f"STAC search succeeded: datetime={dt_expr!r}")
        break
    except Exception as e:
        print(f"  retry: {dt_expr!r} -> {type(e).__name__}")

if items_list:
    print(f"\nSentinel-2 scenes over McLean, IL: {len(items_list)}")
    print("\nMost recent scenes:")
    for item in sorted(items_list, key=lambda x: x.datetime, reverse=True)[:5]:
        print(f"  {item.datetime.strftime('%Y-%m-%d')}  {item.id}")
else:
    print("STAC endpoint unavailable — continuing with demo narrative.")
    print("Sentinel-2 nominally revisits the Corn Belt every ~5 days "
          "at 10 m, yielding ~35 cloud-screened scenes per growing season.")

## 4. NDVI in Production — the One-Liner

On each scene the production pipeline runs a single map-algebra call:

```python
ndvi = RS_MapAlgebra(rast, 'FLOAT32', 'out = (rast[7] - rast[3]) / (rast[7] + rast[3]);')
# where rast[3] = band B04 (Red) and rast[7] = band B08 (NIR)
```

followed by `RS_ZonalStats(ndvi, county_aoi, 'mean')` to aggregate to a
per-(county, scene) average. For this demo we skip the pixel math and
synthesize a realistic seasonal NDVI curve.

In [ ]:
# Reference corn NDVI curve: emergence (May) -> canopy close (Jun) ->
# peak (mid-Jul to early Aug) -> senescence (Sep-Oct)
week_of_year = np.arange(14, 44)        # Apr 1 - Oct 28
ref_ndvi = 0.10 + 0.70 * np.exp(
    -((week_of_year - 29) / 7.0) ** 2    # Gaussian peak at week 29 (~Jul 20)
)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(week_of_year, ref_ndvi, color='#2d7a2d', lw=2.5, label='Climatology (healthy corn)')
ax.fill_between(week_of_year, ref_ndvi, alpha=0.15, color='#2d7a2d')
ax.axvspan(27, 33, alpha=0.10, color='#EF9F27',
           label='Yield-determining window (R1-R4)')
ax.set_xlabel('Week of year')
ax.set_ylabel('NDVI')
ax.set_title('Corn NDVI seasonal signature — the yield signal is in the peak',
             fontsize=12, pad=8)
ax.legend(loc='upper left')
ax.set_ylim(0, 0.95)
plt.tight_layout()
plt.savefig('corn_ndvi_climatology.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. County-Week NDVI Series — Current Year vs. Climatology

For each of the 20 counties, generate a 30-week NDVI series for the
current year, plus the 5-year average (climatology). A few counties are
injected with a drought-stress anomaly to make the signal visible.

In [ ]:
rng = np.random.default_rng(7)

# Counties flagged as stressed in the current season (IA / NE band hit by
# a hypothetical flash drought in early July)
STRESSED = {"Sioux", "Plymouth", "Ida", "Holt", "Cuming", "Clay"}

rows = []
for county, state, lon, lat, acres_k in counties:
    # Climatology = reference curve + small county-specific bias
    bias = rng.normal(0, 0.015)
    climo = np.clip(ref_ndvi + bias, 0, 0.95)

    # Current-year NDVI: climatology + noise + optional stress dip
    noise = rng.normal(0, 0.012, len(week_of_year))
    current = climo + noise
    if county in STRESSED:
        # Dip centered on week 28 (early July), 6-week duration
        stress = -0.18 * np.exp(-((week_of_year - 28) / 4.0) ** 2)
        current = current + stress
    current = np.clip(current, 0, 0.95)

    for w, climo_v, cur_v in zip(week_of_year, climo, current):
        rows.append((
            county, state, int(w),
            float(round(climo_v, 4)),
            float(round(cur_v, 4)),
            float(round(cur_v - climo_v, 4)),
            float(acres_k),
        ))

ndvi_schema = StructType([
    StructField("county",        StringType()),
    StructField("state",         StringType()),
    StructField("week",          StringType()),   # keep as string week-of-year
    StructField("ndvi_climo",    DoubleType()),
    StructField("ndvi_current",  DoubleType()),
    StructField("ndvi_anomaly",  DoubleType()),
    StructField("corn_acres_k",  DoubleType()),
])

ndvi_df = sedona.createDataFrame(
    [(c, s, str(w), cl, cu, an, ac) for c, s, w, cl, cu, an, ac in rows],
    ndvi_schema
)
ndvi_df.createOrReplaceTempView("county_ndvi")
print(f"County-week NDVI observations: {ndvi_df.count():,}")

# Quick visual — the six stressed counties pull away from climatology in July
pdf = ndvi_df.toPandas()
pdf["week"] = pdf["week"].astype(int)

fig, ax = plt.subplots(figsize=(11, 4.5))
for c in pdf["county"].unique():
    sub = pdf[pdf["county"] == c].sort_values("week")
    is_stressed = c in STRESSED
    ax.plot(sub["week"], sub["ndvi_current"],
            color="#E24B4A" if is_stressed else "#888780",
            lw=1.8 if is_stressed else 0.9,
            alpha=0.95 if is_stressed else 0.5,
            label=c if is_stressed else None)
ax.plot(week_of_year, ref_ndvi, color="#2d7a2d", lw=2.5, ls="--",
        label="Healthy climatology")
ax.axvspan(27, 33, alpha=0.08, color="#EF9F27")
ax.set_xlabel("Week of year")
ax.set_ylabel("NDVI")
ax.set_title("Current-season NDVI per county — stressed counties in red",
             fontsize=12, pad=8)
ax.legend(loc="lower center", ncol=4, fontsize=8)
ax.set_ylim(0, 0.95)
plt.tight_layout()
plt.savefig("corn_ndvi_county_series.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Stress Detection — Peak-Window Anomaly by County

The actionable number is the average NDVI anomaly during the yield-
determining window (weeks 27-33). Rank counties by this and you get a
leaderboard of where yield risk is building.

In [ ]:
stress_df = sedona.sql("""
    SELECT
        county, state, corn_acres_k,
        ROUND(AVG(ndvi_current),  3) AS avg_peak_ndvi,
        ROUND(AVG(ndvi_climo),    3) AS avg_peak_climo,
        ROUND(AVG(ndvi_anomaly),  3) AS peak_anomaly,
        CASE
            WHEN AVG(ndvi_anomaly) <= -0.10 THEN 'Severe Stress'
            WHEN AVG(ndvi_anomaly) <= -0.05 THEN 'Elevated Stress'
            WHEN AVG(ndvi_anomaly) <= -0.02 THEN 'Mild Stress'
            WHEN AVG(ndvi_anomaly) <   0.02 THEN 'Normal'
            ELSE                                 'Above Trend'
        END AS stress_tier
    FROM county_ndvi
    WHERE CAST(week AS INT) BETWEEN 27 AND 33
    GROUP BY county, state, corn_acres_k
    ORDER BY peak_anomaly ASC
""").cache()
stress_df.createOrReplaceTempView("county_stress")
stress_df.show(truncate=False)

## 7. The Informational Edge — Satellite Signal vs. USDA Release

Aggregate county-level anomalies into an **acreage-weighted peak-NDVI
anomaly index**, track it weekly against a (synthesized) corn futures
price, and show the satellite anomaly moves weeks before the USDA yield
forecast mark-down.

In [ ]:
weekly_index_df = sedona.sql("""
    SELECT
        CAST(week AS INT) AS week,
        ROUND(SUM(ndvi_anomaly * corn_acres_k) / SUM(corn_acres_k), 4)
            AS weighted_anomaly
    FROM county_ndvi
    GROUP BY week
    ORDER BY week
""").toPandas()

wi = weekly_index_df.sort_values("week").copy()

# Synthesized USDA yield forecast (bu/acre): starts at trend ~180, marks
# down lagged 3-4 weeks behind the satellite anomaly.
wi["usda_yield_bpa"] = (
    180.0 + 120.0 * wi["weighted_anomaly"].rolling(4, min_periods=1).mean()
).shift(3).bfill().round(1)

# Corn futures ($/bushel): anti-correlated with yield; reacts to satellite
# anomaly with ~1 week lag
price_shock = -6.0 * wi["weighted_anomaly"].shift(1).fillna(0)
wi["corn_price"] = (4.60 + price_shock + rng.normal(0, 0.05, len(wi))).round(3)

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(wi["week"], wi["weighted_anomaly"],
         color="#1f77b4", lw=2.2, label="Satellite NDVI anomaly (acreage-weighted)")
ax1.axhline(0, color="#333", lw=0.5)
ax1.set_xlabel("Week of year")
ax1.set_ylabel("NDVI anomaly", color="#1f77b4")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax1.axvspan(27, 33, alpha=0.08, color="#EF9F27")

ax2 = ax1.twinx()
ax2.plot(wi["week"], wi["usda_yield_bpa"],
         color="#888780", lw=1.8, ls="--",
         label="USDA forecast yield (bu/acre, lagged 3 weeks)")
ax2.plot(wi["week"], wi["corn_price"] * 35,
         color="#E24B4A", lw=1.8, alpha=0.9,
         label="Corn futures (scaled)")
ax2.set_ylabel("USDA bu/acre | corn price (scaled)", color="#555")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left", fontsize=9)
plt.title("Satellite stress signal leads USDA by 3+ weeks", fontsize=12, pad=8)
plt.tight_layout()
plt.savefig("corn_signal_vs_usda.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Quantifying the Alpha — Trading Rule

Rule: **go long corn when the weekly NDVI anomaly index turns negative,
flat otherwise.** Compare cumulative P&L to buy-and-hold over the
growing season.

In [ ]:
bt = wi.copy()
bt["price_ret"]    = bt["corn_price"].pct_change().fillna(0)
bt["signal"]       = (bt["weighted_anomaly"].shift(1) < 0).astype(int)
bt["strategy_ret"] = bt["signal"] * bt["price_ret"]
bt["strategy_cum"] = (1 + bt["strategy_ret"]).cumprod() - 1
bt["buyhold_cum"]  = (1 + bt["price_ret"]).cumprod() - 1

weeks_in_signal = int(bt["signal"].sum())
print(f"Weeks long:              {weeks_in_signal} of {len(bt)}")
print(f"Cumulative strategy:     {bt['strategy_cum'].iloc[-1]:+.1%}")
print(f"Cumulative buy-and-hold: {bt['buyhold_cum'].iloc[-1]:+.1%}")

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(bt["week"], bt["strategy_cum"] * 100,
        color="#1f77b4", lw=2.2, label="NDVI-triggered long")
ax.plot(bt["week"], bt["buyhold_cum"] * 100,
        color="#888780", lw=1.6, ls="--", label="Corn buy-and-hold")
ax.axhline(0, color="#333", lw=0.6)
ax.set_xlabel("Week of year")
ax.set_ylabel("Cumulative return (%)")
ax.set_title("Cumulative P&L — NDVI-triggered long vs. passive corn",
             fontsize=12, pad=8)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("corn_signal_backtest.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Ship It — GeoJSON Feed for the Desk Dashboard

Export the county-level stress leaderboard as a GeoJSON
`FeatureCollection` with each county's `stress_tier`, anomaly, and
acreage-weighted TIV-at-risk proxy, ready for a color-coded map.

In [ ]:
stress_geojson_df = sedona.sql("""
    SELECT
        s.county, s.state, s.corn_acres_k,
        s.avg_peak_ndvi, s.peak_anomaly, s.stress_tier,
        c.lon, c.lat,
        ST_AsGeoJSON(c.aoi) AS geojson
    FROM county_stress s
    JOIN counties c USING (county, state)
    ORDER BY s.peak_anomaly ASC
""")

features = [
    {
        "type": "Feature",
        "properties": {
            "county":         row.county,
            "state":          row.state,
            "corn_acres_k":   row.corn_acres_k,
            "avg_peak_ndvi":  row.avg_peak_ndvi,
            "peak_anomaly":   row.peak_anomaly,
            "stress_tier":    row.stress_tier,
        },
        "geometry": json.loads(row.geojson),
    }
    for row in stress_geojson_df.collect()
]
fc = {"type": "FeatureCollection", "features": features}

out_path = "/tmp/corn_belt_stress.geojson"
with open(out_path, "w") as fh:
    json.dump(fc, fh, indent=2)

print(f"Wrote {len(features)} county stress features to {out_path}\n")
for feat in features[:5]:
    p = feat["properties"]
    print(f"  {p['county']:<12} {p['state']}  "
          f"anomaly {p['peak_anomaly']:+.3f}  {p['stress_tier']}")

## 10. Preview on a Map

In [ ]:
map_df = sedona.sql("""
    SELECT
        s.county, s.state, s.corn_acres_k,
        s.peak_anomaly, s.stress_tier,
        c.aoi AS geometry
    FROM county_stress s
    JOIN counties c USING (county, state)
""")
SedonaKepler.create_map(map_df, name="Corn Belt Stress — Peak Anomaly")

---

## From demo to production

Same Wherobots surface, swap in live data:

| Demo step | Production replacement |
|---|---|
| 20 hardcoded county centroids | TIGER/Line county polygons joined to NASS cropland mask |
| Synthesized NDVI series | `RS_MapAlgebra((B08-B04)/(B08+B04))` + `RS_ZonalStats` on Sentinel-2 L2A |
| 5-year climatology | HLS / MODIS 250 m historical archive |
| Synthesized USDA yield | NASS Quick Stats API pull |
| Synthesized corn price | CME/ICE front-month feed |
| Static GeoJSON file | Nightly scheduled job → S3 → frontend map |

## Outputs

| File | Contents |
|---|---|
| `corn_ndvi_climatology.png` | Healthy corn NDVI seasonal curve explainer |
| `corn_ndvi_county_series.png` | Per-county NDVI, stressed counties highlighted |
| `corn_signal_vs_usda.png` | Satellite signal vs. lagged USDA yield + price |
| `corn_signal_backtest.png` | NDVI-triggered long vs. buy-and-hold P&L |
| `/tmp/corn_belt_stress.geojson` | County AOIs with stress tier for dashboard |